# openeo_mountains_snow: a tiny end-to-end run on CDSE

This notebook runs the full chain on a **1.5 x 1.5 km** area in the Senales valley (Italian Alps, 1950-2750 m), for one hydrological year (Oct 2022 - Aug 2023):

1. **Snow map** (Sentinel-2, 20 m): `snow_cover_fraction_cube` -> `snow_map_from_snow_cover_fraction`
2. **MODIS snow cover fraction** (500 m reference): `create_modis_snow_cover_fraction_cube`
3. **Snow cover area (SCA) reconstruction**, gap-filled daily snow maps: `snow_cover_area_cube`
4. **Snow water equivalent (SWE)**: `swe_meteo_cube` (forcing) and `snow_water_equivalent_cube`
5. **Scientific validity**: a few physical and independent-sensor checks, with notes.

Every step is a separate openEO batch job. Downstream steps load the finished upstream result with `load_stac_from_job`. Rebuilding the whole chain in one job runs out of memory, because the Sentinel-2 snow chain is the expensive part. Job ids are stored in `tmp/intro_tiny/jobs.json`, so rerunning a cell resumes the job instead of resubmitting it. To redo a step, delete its entry and its folder.

**Cost:** the snow job is the largest (about 400 credits last time). The others are small.

In [1]:
import json
import re
import sys
import time
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
sys.path.insert(0, str(ROOT / 'src'))

import matplotlib.pyplot as plt
import numpy as np
import openeo
import pandas as pd
import xarray as xr
from pyproj import Transformer
from shapely.geometry import box
from shapely.ops import transform

from openeo_mountains_snow.config import JOB_OPTIONS, build_scf_config
from openeo_mountains_snow.snow_cover_area.conditional_probability import create_modis_snow_cover_fraction_cube
from openeo_mountains_snow.snow_cover_area.snow_cover_area import snow_cover_area_cube
from openeo_mountains_snow.snow_cover_area.snow_map import snow_map_from_snow_cover_fraction
from openeo_mountains_snow.snow_cover_fraction.snow_cover_fraction import snow_cover_fraction_cube
from openeo_mountains_snow.snow_water_equivalent.snow_water_equivalent import (
    basin_snow_cover_area, snow_water_equivalent_cube, swe_meteo_cube,
)

# 1.5 x 1.5 km = 3 x 3 MODIS pixels, aligned to the 500 m grid.
EXTENT = {'west': 634000.0, 'south': 5172500.0, 'east': 635500.0, 'north': 5174000.0, 'crs': 'EPSG:32632'}
PERIOD = ['2022-10-01', '2023-09-01']  # end exclusive
AOI = transform(Transformer.from_crs(EXTENT['crs'], 'EPSG:4326', always_xy=True).transform,
                box(EXTENT['west'], EXTENT['south'], EXTENT['east'], EXTENT['north']))
OUT = ROOT / 'tmp' / 'intro_tiny'
OUT.mkdir(parents=True, exist_ok=True)

# Python UDFs run in the executors' memory overhead; the S2 snow chain needs the most.
OPTIONS = {**JOB_OPTIONS, 'executor-memoryOverhead': '6G'}
OPTIONS_SNOW = {**JOB_OPTIONS, 'executor-memory': '10G', 'executor-memoryOverhead': '8G'}

connection = openeo.connect('https://openeo.dataspace.copernicus.eu/').authenticate_oidc()

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
def _retry(fn, attempts=10):
    """Call fn, backing off when CDSE answers 429 Too Many Requests."""
    for attempt in range(attempts):
        try:
            return fn()
        except (openeo.rest.OpenEoApiError, openeo.rest.OpenEoApiPlainError) as e:
            if e.http_status_code != 429 or attempt == attempts - 1:
                raise
        time.sleep(10 * (attempt + 1))


def _jobs():
    f = OUT / 'jobs.json'
    return json.loads(f.read_text()) if f.exists() else {}


def submit(name, build, out_format='netCDF', options=OPTIONS):
    """Create and start the job `name` (cube from build()), unless jobs.json already has it."""
    ids = _jobs()
    if name not in ids:
        job = _retry(lambda: build().create_job(out_format=out_format, title=f'intro_tiny_{name}', job_options=options))
        ids[name] = job.job_id
        (OUT / 'jobs.json').write_text(json.dumps(ids, indent=1))
    job = connection.job(ids[name])
    if _retry(job.status) == 'created':
        _retry(job.start)
    print(name, ids[name], _retry(job.status))


def fetch(name):
    """Wait for job `name`, download it once, return it as an xarray Dataset."""
    target = OUT / name
    if not (target / 'done').exists():
        job = connection.job(_jobs()[name])
        while (status := _retry(job.status)) not in ('finished', 'error', 'canceled'):
            time.sleep(30)
        if status != 'finished':
            errors = [l.get('message') for l in job.logs(level='error')]
            raise RuntimeError(f'{name} {status}: {errors[-3:]}')
        _retry(lambda: job.get_results().download_files(target))
        (target / 'done').touch()
    return _load(target)


def result(name):
    """Finished result of an upstream job as input cube (must be GTiff: CDSE load_stac cannot read netCDF results)."""
    return connection.load_stac_from_job(_jobs()[name])


def _load(folder):
    ncs, tifs = sorted(folder.glob('*.nc')), sorted(folder.glob('*.tif'))
    if ncs:
        ds = xr.open_mfdataset(ncs, combine='by_coords', mask_and_scale=False) if len(ncs) > 1 else xr.open_dataset(ncs[0], mask_and_scale=False)
        return ds.drop_vars('crs', errors='ignore').load()
    import rasterio  # GTiff results: one file per date, band names in the descriptions
    parts = []
    for f in tifs:
        with rasterio.open(f) as src:
            a, t = src.read(), src.transform
            names = [d or f'band_{i + 1}' for i, d in enumerate(src.descriptions)]
        part = xr.Dataset({n: (('y', 'x'), a[i]) for i, n in enumerate(names)},
                          coords={'x': t.c + t.a * (np.arange(a.shape[2]) + 0.5), 'y': t.f + t.e * (np.arange(a.shape[1]) + 0.5)})
        date = re.search(r'\d{4}-\d{2}-\d{2}', f.name)
        parts.append(part.expand_dims(t=[pd.Timestamp(date.group())]) if date else part)
    return xr.concat(parts, 't') if len(parts) > 1 else parts[0]

## 1. Sentinel-2 snow map

`snow_cover_fraction_cube` runs the SnowFLAKES step on Sentinel-2 L1C: cloud/water mask, local solar incidence angle, shadow mask, then the representative-pixels UDF (SVM trained per scene). `snow_map_from_snow_cover_fraction` binarizes the fraction to the 20 m snow map: **0** no snow, **100** snow, **205** cloud/unknown.

The hosted glacier mask only covers the Andes (RGI 7, UTM 19S), so it is switched off for this Alpine AOI. With the mask on, `load_stac` would find no data here.

The independent jobs (snow, MODIS, meteo, terrain) are all submitted here so they run in parallel.

In [ ]:
cfg = build_scf_config()
cfg.glacier_mask.stac_url = None  # glacier mask covers the Andes only

submit('snow', lambda: snow_map_from_snow_cover_fraction(snow_cover_fraction_cube(AOI, PERIOD, connection, cfg)),
       out_format='GTiff', options=OPTIONS_SNOW)  # GTiff: loaded again by the SCA job
submit('modis_scf', lambda: create_modis_snow_cover_fraction_cube(connection, PERIOD, EXTENT))
submit('meteo', lambda: swe_meteo_cube(connection, EXTENT, PERIOD[0], PERIOD))  # only for the checks; the SWE job builds its own
submit('terrain', lambda: connection.load_collection('COPERNICUS_30', spatial_extent=EXTENT)
       .resample_spatial(resolution=20, projection=32632, method='bilinear'))

In [ ]:
snow = fetch('snow')['snow'].transpose('t', 'y', 'x')
share = pd.DataFrame({'snow': (snow == 100).mean(['x', 'y']), 'no snow': (snow == 0).mean(['x', 'y']),
                      'cloud/unknown': (snow == 205).mean(['x', 'y'])}, index=snow.t.to_index())
print(f'{snow.sizes["t"]} Sentinel-2 dates, {dict(snow.sizes)}; dates with any clear pixel: {int((share["cloud/unknown"] < 1).sum())}')

fig, axes = plt.subplots(1, 5, figsize=(16, 3))
clear_dates = share.index[share['cloud/unknown'] < 0.5]
for ax, t in zip(axes, clear_dates[np.linspace(0, len(clear_dates) - 1, 5).astype(int)] if len(clear_dates) else []):
    ax.imshow(snow.sel(t=t), cmap=plt.matplotlib.colors.ListedColormap(['sienna', 'white', 'grey']),
              norm=plt.matplotlib.colors.BoundaryNorm([-1, 50, 150, 256], 3))
    ax.set_title(f'{t:%Y-%m-%d}'); ax.set_xticks([]); ax.set_yticks([])
fig.suptitle('Snow map on mostly clear dates (white snow, brown no snow, grey cloud)')
share.plot.area(figsize=(14, 2.5), title='Snow map composition per Sentinel-2 date', color=['tab:blue', 'sienna', 'lightgrey']);

## 2. MODIS snow cover fraction

`create_modis_snow_cover_fraction_cube` loads MOD10A1 NDSI snow cover (Terra, ~10:30, S2 passes at ~10:20), converts NDSI to snow cover fraction (0-100 %, 205 = cloud/flag) and puts it on the 20 m grid. It feeds the SCA reconstruction and is our independent reference for the S2 snow map.

In [ ]:
modis = fetch('modis_scf')['scf'].transpose('t', 'y', 'x')
modis_clear = (modis <= 100).mean(['x', 'y'])
print(f'{modis.sizes["t"]} MODIS dates in {PERIOD}, of which >= 50 % clear: {int((modis_clear >= 0.5).sum())}')
fig, ax = plt.subplots(figsize=(14, 2.5))
ax.plot(modis.t, modis.where(modis <= 100).mean(['x', 'y']), 'o', ms=3)
ax.set_ylabel('AOI-mean SCF (%)'); ax.set_title('MODIS snow cover fraction (clear pixels)');

## 3. Snow cover area reconstruction

`snow_cover_area_cube` takes the snow map, builds the MODIS SCF and the per-pixel conditional probabilities (`cp_*`, `occ_*`: how often a pixel is snow at a given MODIS SCF), and runs the reconstruction UDF. The UDF fills cloudy/missing days. Output: one map per day, 0 / 100 / 205.

In [ ]:
submit('sca', lambda: snow_cover_area_cube(connection, result('snow'), EXTENT, PERIOD), out_format='GTiff')  # GTiff: loaded by SWE
sca_ds = fetch('sca')
sca = sca_ds[list(sca_ds.data_vars)[0]].transpose('t', 'y', 'x')
steps = pd.Series(np.diff(sca.t.values).astype('timedelta64[D]').astype(int)).value_counts().sort_index()
print(f'{sca.sizes["t"]} SCA dates; time steps (days): {steps.to_dict()}')

fig, axes = plt.subplots(1, 2, figsize=(15, 3), gridspec_kw={'width_ratios': [3, 1]})
axes[0].plot(snow.t, (snow == 205).mean(['x', 'y']), 'x', label='input: cloud/unknown (S2 dates)')
axes[0].plot(sca.t, (sca == 205).mean(['x', 'y']), label='reconstructed: unknown left')
axes[0].plot(sca.t, (sca == 100).mean(['x', 'y']), label='reconstructed: snow share')
axes[0].legend(fontsize=8); axes[0].set_title('Reconstructed SCA')
axes[1].imshow((sca == 100).sum('t'), cmap='Blues'); axes[1].set_title('days with snow'); axes[1].set_xticks([]); axes[1].set_yticks([])
plt.tight_layout()

## 4. Snow water equivalent

`snow_water_equivalent_cube` downscales AGERA5 meteo to the 30 m DEM with `swe_meteo_cube`: temperature with monthly lapse rates, precipitation, and shortwave corrected for slope/aspect and sun position. It then runs the SWE UDF. The UDF works backwards from each melt-out date: it sums the modelled melt and spreads it over the accumulation days. `basin_sca` (the AOI-mean SCA per day, from the downloaded SCA) decides accumulation on warm days, as in the original method.

In [ ]:
basin_sca = basin_snow_cover_area(sca)
submit('swe', lambda: snow_water_equivalent_cube(connection, result('sca'), EXTENT, PERIOD, basin_sca=basin_sca))
swe_ds = fetch('swe')
swe = swe_ds['swe'].transpose('t', 'y', 'x')
met = fetch('meteo')

T_THR = 2.0  # SWE UDF temperature_threshold: precipitation counts as snow below this daily-mean T (°C)
ta, pr = met['temperature_downscaled'].mean(['x', 'y']), met['precipitation'].mean(['x', 'y'])
snowfall = pr.where(ta < T_THR, 0)
swe_mean = swe.mean(['x', 'y'])
print(f'SWE: {swe.sizes["t"]} dates, NaN share {float(swe.isnull().mean()):.2f}, AOI-mean peak {float(swe_mean.max()):.0f} mm '
      f'on {str(swe_mean.idxmax().values)[:10]} | snowfall (T < {T_THR} °C) {float(snowfall.sum()):.0f} mm')

fig, ax = plt.subplots(figsize=(14, 3.5))
ax.plot(swe_mean.t, swe_mean, label='AOI-mean SWE')
ax.plot(snowfall.t, snowfall.cumsum('t'), 'k--', lw=1, label='cumulative snowfall (upper bound)')
ax.set_ylabel('mm'); ax.legend(); ax.set_title('SWE vs the snowfall that could have built it');

## 5. Scientific validity

There is no in-situ data for this AOI, so the checks are of three kinds:
- **Independent sensor:** the S2 snow map should match MODIS on the same day.
- **Physical pattern:** more snow higher up, longer on north faces.
- **Conservation:** peak SWE cannot exceed the snowfall before it.

Each check adds a PASS / FAIL row (CHECK = rough literature range) to the scorecard at the end.

In [ ]:
checks = []


def check(step, name, value, expected, ok):
    checks.append({'step': step, 'check': name, 'value': value, 'expected': expected,
                   'result': 'CHECK' if ok is None else ('PASS' if ok else 'FAIL')})


def spearman(a, b):
    m = np.isfinite(a) & np.isfinite(b)
    return pd.Series(a[m]).rank().corr(pd.Series(b[m]).rank())


def on_grid(a, grid):
    """Nearest pixel of `a` for each pixel of the UTM 32N `grid` (a may be on a lat/lon grid)."""
    gx, gy = np.meshgrid(grid.x.values, grid.y.values)
    if abs(float(a.x[0])) < 1000:  # lat/lon
        gx, gy = Transformer.from_crs('EPSG:32632', 'EPSG:4326', always_xy=True).transform(gx, gy)
    pts = dict(x=xr.DataArray(gx, dims=('y', 'x')), y=xr.DataArray(gy, dims=('y', 'x')))
    return a.sel(**pts, method='nearest').assign_coords(x=grid.x.values, y=grid.y.values).astype(float)


# Terrain on the snow grid; aspect = downslope direction, clockwise from north.
dem = fetch('terrain')
elev = on_grid(dem[list(dem.data_vars)[0]].squeeze(drop=True), snow)
dzdy, dzdx = np.gradient(elev.values, 20, 20)  # rows go north -> south
aspect = np.degrees(np.arctan2(-dzdx, dzdy)) % 360
facing = np.where((aspect >= 315) | (aspect < 45), 'north', np.where((aspect >= 135) & (aspect < 225), 'south', 'east/west'))
print(f'elevation {float(elev.min()):.0f}..{float(elev.max()):.0f} m | facing: {pd.Series(facing.ravel()).value_counts().to_dict()}')

# 5.1 S2 snow map vs MODIS on common dates
s2, mo = xr.align(snow, modis, join='inner')
s2_clear, mo_clear = s2.isin([0, 100]), mo <= 100
day = pd.DataFrame({'s2_clear': s2_clear.mean(['x', 'y']), 'mo_clear': mo_clear.mean(['x', 'y']),
                    's2_snow': (s2 == 100).sum(['x', 'y']) / s2_clear.sum(['x', 'y']),
                    'mo_snow': mo.where(mo_clear).mean(['x', 'y']) / 100}, index=s2.t.to_index())
both = day[(day.s2_clear >= 0.5) & (day.mo_clear >= 0.5)]
mo_clear_days = day[day.mo_clear >= 0.5]
lost = mo_clear_days[mo_clear_days.s2_clear < 0.05]
bias, r = (both.s2_snow - both.mo_snow).mean(), both.s2_snow.corr(both.mo_snow)
check('snow map', 'snow fraction vs MODIS: bias', f'{bias:+.2f} (n={len(both)})', '|bias| < 0.15', abs(bias) < 0.15)
check('snow map', 'snow fraction vs MODIS: correlation', f'r = {r:.2f}', 'r > 0.7', r > 0.7)
check('snow map', 'S2 all cloud while MODIS clear', f'{len(lost)} / {len(mo_clear_days)} days', '< 20 %', len(lost) < 0.2 * len(mo_clear_days))

# 5.2 Physical patterns of the S2 snow map in spring (Mar-May)
spring = snow.sel(t=snow.t.dt.month.isin([3, 4, 5]))
freq = (spring == 100).sum('t') / spring.isin([0, 100]).sum('t')  # per-pixel snow frequency of clear obs
band = (elev // 100) * 100
by_band = freq.groupby(band.rename('band')).mean()
by_face = pd.Series(freq.values.ravel()).groupby(facing.ravel()).mean()
rho = spearman(by_band.band.values.astype(float), by_band.values)
check('snow map', 'spring snow increases with elevation', f'rank corr {rho:.2f}', '> 0', rho > 0)
check('snow map', 'spring snow north > south', f'N {by_face.get("north", np.nan):.2f} / S {by_face.get("south", np.nan):.2f}',
      'N > S', by_face.get('north', 0) > by_face.get('south', 0))

# 5.3 SCA reconstruction
sca_on, s2_on = xr.align(sca, snow, join='inner')
keep = s2_on.isin([0, 100])
fidelity = float((sca_on == s2_on).where(keep).mean())
unknown_in = float((~snow.isin([0, 100])).mean())
unknown_out = float((sca == 205).mean())
v = sca.where(sca != 205).ffill('t').values
flips = ((np.diff(v, axis=0) != 0) & np.isfinite(np.diff(v, axis=0))).sum(0)
check('SCA', 'keeps clear S2 observations', f'{fidelity:.4f}', '1.0', fidelity > 0.999)
check('SCA', 'unknown (205) left', f'{unknown_out:.2f} (S2 input {unknown_in:.2f})', '< 0.10', unknown_out < 0.10)
check('SCA', 'daily, gap-free time axis', f'{steps.to_dict()}', 'all steps 1 day', list(steps.index) == [1])
check('SCA', 'snow/no-snow flips per pixel', f'median {np.median(flips):.0f}', '<= 6', np.median(flips) <= 6)

# 5.4 SWE
peak_t = swe_mean.idxmax().values
sf_before = float(snowfall.sel(t=slice(None, peak_t)).sum())
peak_px = on_grid(swe.max('t'), snow)
rho_swe = spearman(elev.values.ravel(), peak_px.values.ravel())
check('SWE', 'NaN share', f'{float(swe.isnull().mean()):.2f}', '< 0.05', float(swe.isnull().mean()) < 0.05)
check('SWE', 'mass: AOI peak <= snowfall before peak', f'{float(swe_mean.max()):.0f} / {sf_before:.0f} mm', '<= 1', float(swe_mean.max()) <= sf_before)
check('SWE', 'AOI-mean peak SWE', f'{float(swe_mean.max()):.0f} mm', '~100-400 mm (low-snow year, rough)', None)
check('SWE', 'peak SWE increases with elevation', f'rank corr {rho_swe:.2f}', '> 0', rho_swe > 0)

fig, axes = plt.subplots(1, 4, figsize=(18, 3.8))
im = axes[0].imshow(elev, cmap='terrain'); axes[0].set_title('elevation (m)'); plt.colorbar(im, ax=axes[0])
axes[1].scatter(both.mo_snow, both.s2_snow); axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].set_xlabel('MODIS snow fraction'); axes[1].set_ylabel('S2 snow fraction'); axes[1].set_title(f'S2 vs MODIS (n={len(both)})')
im = axes[2].imshow(freq, cmap='Blues', vmin=0, vmax=1); axes[2].set_title('spring snow frequency (S2)'); plt.colorbar(im, ax=axes[2])
im = axes[3].imshow(peak_px, cmap='Blues'); axes[3].set_title('peak SWE (mm)'); plt.colorbar(im, ax=axes[3])
for ax in (axes[0], axes[2], axes[3]):
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()

score = pd.DataFrame(checks)
print(score.result.value_counts().to_dict())
with pd.option_context('display.max_colwidth', None):
    display(score)

### Notes on validity

*To be written from the results of this run.*